In [207]:
import pinecone
import langchain
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import OllamaLLM
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import PromptTemplate


In [208]:
import os

In [209]:

file_loader=PyPDFDirectoryLoader("./")
documents=file_loader.load()


In [ ]:

len(documents)
print(documents)

In [211]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
    separators=['\n','.',',',' ']
)
chunks=splitter.split_documents(documents)
print(len(chunks))



22


In [ ]:
embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
os.environ["PINECONE_API_KEY"]="ur api key here"



In [213]:
vector_store=PineconeVectorStore.from_documents(chunks,embeddings,index_name="langchainproject")

In [214]:
##consine similarity retrieve results from vectorDB
def retrieve_query(query,k=3):
    matching_results=vector_store.similarity_search(query,k=3)
    return matching_results

In [215]:
llm=OllamaLLM(model='llama3.2',temperature=0.7)
prompt = PromptTemplate(
    input_variables=['question', 'context'],
    template="Context:\n{context}\n\nQuestion: {question}\nAnswer based only on the context above."
)



In [216]:
def retrieve_answer(query):
    doc_search=retrieve_query(query)
    print(doc_search)
    context="\n".join([documents.page_content for documents in  doc_search])
    chain= prompt | llm
    print(context)
    return chain.invoke({"context":context,"question":query})


In [ ]:
docs = retrieve_query("skills")
context = "\n".join([doc.page_content for doc in docs])
print(context[:500])  # check what context looks like

In [ ]:
query = "what are soumil's skills"
answer = retrieve_answer(query)
print(answer)